In [0]:
import os
import sys

from pyspark.sql.functions import *
from pyspark.sql.types import *

pth = os.path.join(os.getcwd(), '..', '..')
sys.path.append(pth)
print(pth)

from utils.transformations import reusable

print("Successfully imported Python module: transformations.py")

### **DimUser**

In [0]:
df=spark.read.format("parquet")\
        .load("abfss://bronze@azuresaiprojects.dfs.core.windows.net/DimUser")

In [0]:
display(df)

### **AutoLoader**

In [0]:
df_user=spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format", "parquet")\
        .option("cloudFiles.schemaLocation", "abfss://silver@azuresaiprojects.dfs.core.windows.net/DimUser/checkpoint")\
        .option("schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@azuresaiprojects.dfs.core.windows.net/DimUser")


In [0]:
display(df_user)

In [0]:
df_user_obj = reusable()

df_user_testdropcolumns = df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user_testdropDuplicates = df_user.dropDuplicates(['user_id'])
display(df_user_testdropcolumns)
display(df_user_testdropDuplicates)

In [0]:
df_user.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", "abfss://silver@azuresaiprojects.dfs.core.windows.net/DimUser/checkpoint")\
        .trigger(once=True)\
        .option("path","abfss://silver@azuresaiprojects.dfs.core.windows.net/DimUser/data")\
        .toTable("spotifycatalog.silver.DimUser")

### **DimArtist**

In [0]:
df_art=spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format", "parquet")\
        .option("cloudFiles.schemaLocation", "abfss://silver@azuresaiprojects.dfs.core.windows.net/DimArtist/checkpoint")\
        .option("schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@azuresaiprojects.dfs.core.windows.net/DimArtist")

In [0]:
display(df_art)

In [0]:
df_art_obj = reusable()

df_art_testdropColumns = df_art_obj.dropColumns(df_art,['_rescued_data'])
df_art = df_art.dropDuplicates(['artist_id'])
display(df_art_testdropColumns)
display(df_art)

In [0]:
df_art.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", "abfss://silver@azuresaiprojects.dfs.core.windows.net/DimArtist/checkpoint")\
        .trigger(once=True)\
        .option("path","abfss://silver@azuresaiprojects.dfs.core.windows.net/DimArtist/data")\
        .toTable("spotifycatalog.silver.DimArtist")

### **DimTrack**

In [0]:
df_track=spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@azuresaiprojects.dfs.core.windows.net/DimTrack/checkpoint")\
        .option("schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@azuresaiprojects.dfs.core.windows.net/DimTrack")

In [0]:
display(df_track)

In [0]:
df_track = df_track.withColumn("durationFlag", when(col('duration_sec')<150, "low")
                               .when((col('duration_sec')>=150) & (col('duration_sec')<300), "medium")
                               .when(col('duration_sec')>=300, "high"))
df_track = df_track.withColumn("track_name",regexp_replace('track_name','-',' '))
                                                           
df_track = reusable().dropColumns(df_track,['_rescued_data'])

display(df_track)

In [0]:
df_track.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation","abfss://silver@azuresaiprojects.dfs.core.windows.net/DimTrack/checkpoint")\
        .trigger(once=True)\
        .option("path","abfss://silver@azuresaiprojects.dfs.core.windows.net/DimTrack/data")\
        .toTable("spotifycatalog.silver.DimTrack")


### **DimDate**

In [0]:
df_date=spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format", "parquet")\
        .option("cloudFiles.schemaLocation", "abfss://silver@azuresaiprojects.dfs.core.windows.net/DimDate/checkpoint")\
        .option("schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@azuresaiprojects.dfs.core.windows.net/DimDate")

In [0]:
display(df_date)


In [0]:
df_date_obj = reusable()

df_date1 = df_date_obj.dropColumns(df_date,['_rescued_data'])
df_date2 = df_date.dropDuplicates(['date_key'])

display(df_date1)
display(df_date2)

In [0]:
df_date.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", "abfss://silver@azuresaiprojects.dfs.core.windows.net/DimDate/checkpoint")\
        .trigger(once=True)\
        .option("path","abfss://silver@azuresaiprojects.dfs.core.windows.net/DimDate/data")\
        .toTable("spotifycatalog.silver.DimDate")

### **FactStream**

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@azuresaiprojects.dfs.core.windows.net/FactStream/checkpoint")\
        .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@azuresaiprojects.dfs.core.windows.net/FactStream")

In [0]:
display(df_fact)

In [0]:
df_fact = reusable().dropColumns(df_fact,['_rescued_data'])

display(df_fact)

In [0]:
df_fact.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation","abfss://silver@azuresaiprojects.dfs.core.windows.net/FactStream/checkpoint")\
        .trigger(once=True)\
        .option("path","abfss://silver@azuresaiprojects.dfs.core.windows.net/FactStream/data")\
        .toTable("spotifycatalog.silver.FactStream")